# Multi-Genie Orchestration

## Parallel Queries and Synthesized Cross-Domain Insights

This notebook walks through the full multi-agent pipeline for the AI Data Analyst Workshop.

**What you'll see:**
1. **End-to-end pipeline** — the full plan → query → synthesize → report flow
2. **Under the hood: LangGraph concepts** — stateful agents, cyclic graphs, and tool routing
3. **Deep-dive step-through** — each pipeline stage executed individually
4. **Two orchestration patterns** — linear pipeline vs. cyclic agent graph

### Pipeline stages
| Stage | Agent | What it does |
|-------|-------|--------------|
| Planning | `PlannerAgent` | Decomposes the question into domain-specific sub-queries |
| Querying | `MultiGenieOrchestrator` | Executes parallel queries across Genie Spaces |
| Synthesizing | `SynthesizerAgent` | Generates cross-domain insights and correlations |
| Reporting | `ReportWriter` | Produces Markdown and HTML dashboard outputs |

## 1. Setup and Dependencies

In [ ]:
# Install dependencies (run once)
%pip install databricks-sdk>=0.40.0 databricks-langchain>=0.13.0 langgraph>=0.2.0 langchain-core>=0.3.0 pydantic>=2.0.0 python-dotenv>=1.0.0 jinja2>=3.0.0 -q

In [ ]:
# Restart Python to pick up new packages (Databricks)
dbutils.library.restartPython()

In [ ]:
# Add src to path for imports
import os
import sys

# For Databricks notebooks
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    # Get the workspace path
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    # For local development
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

## 2. Configuration Widgets

Run the cell below to create interactive configuration widgets.
Use the widgets at the top of the notebook to configure your demo settings.

In [ ]:
# Create Databricks widgets for configuration
# These appear at the top of the notebook for easy access

# Check if running in Databricks
import os

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_DATABRICKS:
    # Remove existing widgets to avoid duplicates
    try:
        dbutils.widgets.removeAll()
    except:
        pass

    # === Genie Space Configuration ===
    dbutils.widgets.text("genie_space_sales_id", "", "1. Sales Space ID")
    dbutils.widgets.text("genie_space_customers_id", "", "2. Customers Space ID")
    dbutils.widgets.text("genie_space_inventory_id", "", "3. Inventory Space ID")

    # === Infrastructure ===
    dbutils.widgets.text("warehouse_id", "", "4. Warehouse ID")
    dbutils.widgets.dropdown(
        "model_endpoint",
        "databricks-meta-llama-3-3-70b-instruct",
        [
            "databricks-meta-llama-3-3-70b-instruct",
            "databricks-meta-llama-3-1-405b-instruct",
            "databricks-dbrx-instruct",
        ],
        "5. Model Endpoint",
    )

    # === Demo Settings ===
    dbutils.widgets.dropdown("mock_mode", "true", ["true", "false"], "6. Mock Mode")
    dbutils.widgets.dropdown("cache_enabled", "true", ["true", "false"], "7. Cache Enabled")
    dbutils.widgets.dropdown(
        "report_type", "Q4 Executive Report", ["Q4 Executive Report", "YTD Summary", "Custom Query"], "8. Report Type"
    )
    dbutils.widgets.text("custom_query", "", "9. Custom Query (if Report Type = Custom)")

    print("Widgets created! Configure settings using the dropdowns at the top of the notebook.")
    print("\nFor Mock Mode demo, leave Space IDs empty and set Mock Mode = true")
else:
    print("Running locally - using environment variables for configuration")
    print("Set: MOCK_MODE=true for demo without Genie access")

In [ ]:
# Read configuration from widgets (Databricks) or environment variables (local)
from src.agents.multi_genie_orchestrator import GenieSpaceConfig
from src.config import Config, clear_config_cache

clear_config_cache()

if IN_DATABRICKS:
    # Read from Databricks widgets
    sales_space_id = dbutils.widgets.get("genie_space_sales_id") or "mock-sales-space"
    customers_space_id = dbutils.widgets.get("genie_space_customers_id") or "mock-customers-space"
    inventory_space_id = dbutils.widgets.get("genie_space_inventory_id") or "mock-inventory-space"
    warehouse_id = dbutils.widgets.get("warehouse_id")
    model_endpoint = dbutils.widgets.get("model_endpoint")
    mock_mode = dbutils.widgets.get("mock_mode") == "true"
    cache_enabled = dbutils.widgets.get("cache_enabled") == "true"
    report_type = dbutils.widgets.get("report_type")
    custom_query = dbutils.widgets.get("custom_query")
else:
    # Read from environment variables (local development)
    sales_space_id = os.getenv("GENIE_SPACE_SALES_ID", "mock-sales-space")
    customers_space_id = os.getenv("GENIE_SPACE_CUSTOMERS_ID", "mock-customers-space")
    inventory_space_id = os.getenv("GENIE_SPACE_INVENTORY_ID", "mock-inventory-space")
    warehouse_id = os.getenv("WAREHOUSE_ID", "")
    model_endpoint = os.getenv("MODEL_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")
    mock_mode = os.getenv("MOCK_MODE", "true").lower() == "true"
    cache_enabled = os.getenv("CACHE_ENABLED", "true").lower() == "true"
    report_type = os.getenv("REPORT_TYPE", "Q4 Executive Report")
    custom_query = os.getenv("CUSTOM_QUERY", "")

# Create main configuration
config = Config(
    genie_space_id=sales_space_id,
    warehouse_id=warehouse_id,
    model_endpoint=model_endpoint,
    mock_mode=mock_mode,
    cache_enabled=cache_enabled,
)

# Configure Genie Spaces
space_configs = [
    GenieSpaceConfig(
        space_id=sales_space_id,
        name="Sales",
        domain="sales, revenue, orders, transactions",
    ),
    GenieSpaceConfig(
        space_id=customers_space_id,
        name="Customers",
        domain="customers, segments, demographics, retention",
    ),
    GenieSpaceConfig(
        space_id=inventory_space_id,
        name="Inventory",
        domain="inventory, stock, products, warehouse",
    ),
]

# Determine question based on report type
REPORT_QUESTIONS = {
    "Q4 Executive Report": "Generate a Q4 executive report analyzing sales performance, customer trends, and inventory status",
    "YTD Summary": "Provide a year-to-date summary of business performance including revenue trends, customer acquisition, and operational metrics",
    "Custom Query": custom_query or "Analyze the current business state",
}
question = REPORT_QUESTIONS.get(report_type, REPORT_QUESTIONS["Q4 Executive Report"])

# Display configuration summary
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"\nMode: {'MOCK (Demo)' if mock_mode else 'LIVE (Real Genie)'}")
print(f"Cache: {'Enabled' if cache_enabled else 'Disabled'}")
print(f"Model: {model_endpoint}")
print("\nGenie Spaces:")
for sc in space_configs:
    status = "mock" if "mock" in sc.space_id else "configured"
    print(f"  - {sc.name} ({sc.domain[:30]}...) [{status}]")
print(f"\nReport Type: {report_type}")
print(f"Question: {question[:80]}{'...' if len(question) > 80 else ''}")

## Why Multiple Genie Spaces?

Real organisations split data across **domain-specific Genie Spaces** for three reasons:

1. **Organisational boundaries** — each team owns its own data and governs access independently.
2. **Domain-specific tuning** — the Genie model is fine-tuned per space, so a Sales space understands revenue semantics while an Inventory space understands stock-level semantics.
3. **Scale** — smaller, focused spaces answer faster and more accurately than one giant space.

### Current Genie Spaces (Velocity Motors)

| Space | Domain | Key Tables |
|-------|--------|------------|
| **Sales** | Revenue, orders, transactions | `sales_transactions`, `products`, `dealerships` |
| **Customers** | Segments, demographics, retention | `customers`, `customer_interactions` |
| **Inventory** | Stock levels, products, warehouse | `inventory_snapshots`, `products` |

> **Note:** Velocity Motors also has an Operations domain (service orders, parts, suppliers). Adding a 4th space requires only one additional `GenieSpaceConfig` — the pipeline scales automatically.

## 3. End-to-End Pipeline

Execute the complete multi-agent pipeline in a single cell with progress tracking.

In [ ]:
from IPython.display import HTML, Markdown, display

from src.agents.multi_genie_orchestrator import MultiGenieOrchestrator
from src.agents.planner_agent import PlannerAgent
from src.agents.report_writer import ReportWriter
from src.agents.synthesizer_agent import SynthesizerAgent
from src.demo import PipelineStage, PipelineState, render_for_notebook, render_progress_html

# Initialize pipeline state
state = PipelineState()
state.initialize_spaces([sc.name for sc in space_configs])

# Initialize agents with configuration from widgets
planner = PlannerAgent(config, space_configs)
orchestrator = MultiGenieOrchestrator(
    space_configs,
    config,
    progress_callback=state.get_progress_callback(),
)
synthesizer = SynthesizerAgent(config)
report_writer = ReportWriter(config)

print(f"Question: {question}")
print("=" * 80)
print()

try:
    # Stage 1: Planning
    state.start_stage(PipelineStage.PLANNING)
    print("[1/4] Planning - Decomposing question into domain-specific queries...")
    state.plan = planner.decompose(question)
    state.complete_stage(PipelineStage.PLANNING)
    print(f"       Created {len(state.plan.sub_queries)} sub-queries targeting: {', '.join(state.plan.target_spaces)}")
    print()

    # Stage 2: Querying
    state.start_stage(PipelineStage.QUERYING)
    print("[2/4] Querying - Executing parallel queries across Genie Spaces...")
    state.multi_result = orchestrator.query_all(question)
    state.reconcile_from_result(state.multi_result)
    state.complete_stage(PipelineStage.QUERYING)

    successful = state.multi_result.successful_results()
    failed = state.multi_result.get_failed_spaces()
    print(f"       Successful: {len(successful)}/{len(space_configs)} spaces")
    if failed:
        print(f"       Failed: {', '.join(failed)}")
    print()

    # Stage 3: Synthesizing
    state.start_stage(PipelineStage.SYNTHESIZING)
    print("[3/4] Synthesizing - Generating cross-domain insights...")
    state.synthesis_result = synthesizer.synthesize(state.multi_result, question)
    state.complete_stage(PipelineStage.SYNTHESIZING)
    print(
        f"       Generated {len(state.synthesis_result.key_insights)} insights, {len(state.synthesis_result.recommendations)} recommendations"
    )
    print()

    # Stage 4: Reporting
    state.start_stage(PipelineStage.REPORTING)
    print("[4/4] Reporting - Generating Markdown and HTML reports...")
    state.markdown_report = report_writer.generate_markdown(state.synthesis_result, title=report_type)
    state.html_report = report_writer.generate_html(state.synthesis_result, title=report_type)
    state.complete_stage(PipelineStage.REPORTING)
    print("       Reports generated successfully!")
    print()

    # Display final progress with HTML cards
    print("=" * 80)
    cache_stats = orchestrator.get_cache_stats()
    display(HTML(render_progress_html(state, cache_stats=cache_stats)))

except Exception as e:
    state.fail_stage(state.current_stage, str(e))
    print(f"\nPipeline failed: {e}")
    display(Markdown(render_for_notebook(state)))

In [ ]:
# Display the generated Markdown report
if state.markdown_report:
    display(Markdown(state.markdown_report))
else:
    print("No report generated. Run the End-to-End cell above first.")

## 4. Under the Hood — LangGraph Concepts

The linear pipeline above is great for **batch reporting**, but what about **interactive, multi-turn** data analysis?

LangGraph lets us build a **cyclic agent graph** — a supervisor that iteratively routes questions to tools, inspects results, and decides what to do next.

### Key concept: `StateGraph`

A LangGraph `StateGraph` is a directed graph where:
- **Nodes** are Python functions (agents, tools, routers)
- **Edges** can be conditional — the next node depends on the current state
- **State** is a shared `TypedDict` passed through every node

```python
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]  # append semantics
    next_agent: str      # routing decision
    genie_result: str    # latest data result
    rag_result: str      # latest document result
    iteration_count: int # safety limit
```

The `Annotated[..., operator.add]` pattern means each node **appends** to the message list rather than replacing it — this is how conversation history accumulates.

In [ ]:
# Create a supervisor agent (LangGraph StateGraph under the hood)
from src.agents.supervisor import create_simple_supervisor

supervisor = create_simple_supervisor(config, thread_id="demo-thread-01")
graph = supervisor.graph

# Inspect the compiled graph
print("Graph nodes:", list(graph.get_graph().nodes))
print(f"Mock mode:  {config.mock_mode}")
print(f"Thread ID:  demo-thread-01")

In [ ]:
# Visualize the graph as a Mermaid diagram
# Copy the output into https://mermaid.live to render it
mermaid_code = graph.get_graph().draw_mermaid()
print(mermaid_code)

### The Cycle Explained

Unlike the linear pipeline (plan → query → synthesize → report), the supervisor graph forms a **cycle**:

```
              ┌─────────────────────────────────┐
              │                                 │
              v                                 │
     ┌──────────────┐     ┌───────────┐     ┌─────────────┐
     │  supervisor  │─────│ route_next│─────│    tools    │
     │  (LLM node)  │     │ (cond.)   │     │ query_data  │
     └──────────────┘     └───────────┘     │ search_docs │
           ^                                 └─────────────┘
           │                                       │
           └───────── after_tools ──────────────┘
```

| Node | Role |
|------|------|
| **supervisor** | LLM-powered decision maker — reads the conversation, decides which tool to call (or to finish) |
| **route_next** | Conditional edge — inspects `next_agent` in state and routes to the right tool node |
| **tools** | Executes `query_data` (Genie) or `search_documents` (RAG) |
| **after_tools** | Loops back to **supervisor** for the next decision |

**Safety limit:** 5 iterations max — prevents infinite loops if the LLM keeps requesting tools.

**Why cycles matter:** The supervisor can inspect a tool’s result and decide to call *another* tool, refine a query, or combine results before responding. This is impossible in a linear pipeline.

In [ ]:
# Query 1: Data question → routes to query_data tool (Genie)
print("=" * 60)
print("QUERY 1: Data question")
print("=" * 60)
response1 = supervisor.query("What are our top selling products by revenue?", verbose=True)
print(f"\nFinal response:\n{response1}")

# Query 2: Document question → routes to search_documents tool (RAG)
print("\n" + "=" * 60)
print("QUERY 2: Document question")
print("=" * 60)
response2 = supervisor.query("What is the company return policy?", verbose=True)
print(f"\nFinal response:\n{response2}")

In [ ]:
# Stateful conversation: the supervisor remembers previous turns
from src.agents.supervisor import create_simple_supervisor

# Fresh supervisor with a new thread
supervisor_stateful = create_simple_supervisor(config, thread_id="demo-thread-02")

# Turn 1
print("TURN 1")
print("-" * 40)
r1 = supervisor_stateful.query("What are the top 3 products by revenue?")
print(r1)

# Turn 2 — follow-up that relies on conversation history
print("\nTURN 2 (follow-up)")
print("-" * 40)
r2 = supervisor_stateful.query("Now show me inventory levels for those same products")
print(r2)

# Inspect the message history
print("\nCONVERSATION HISTORY")
print("-" * 40)
history = supervisor_stateful.get_history()
for msg in history:
    role = type(msg).__name__.replace('Message', '')
    preview = msg.content[:80] + ('...' if len(msg.content) > 80 else '')
    print(f"  {role:10s} | {preview}")

# Comparison
print("\n" + "=" * 60)
print("PATTERN COMPARISON")
print("=" * 60)
print(f"{"Linear Pipeline (Sec. 3)":30s} | {"Cyclic Agent Graph (Sec. 4)":30s}")
print("-" * 63)
print(f"{"Stateless (batch)":30s} | {"Stateful (multi-turn)":30s}")
print(f"{"Fixed stage order":30s} | {"Dynamic tool routing":30s}")
print(f"{"All spaces queried":30s} | {"Only relevant tools called":30s}")
print(f"{"Best for reports":30s} | {"Best for exploration":30s}")

## 5. Deep Dive: Step-Through Mode

Execute each pipeline stage individually to understand the data flow.

### 5.1 Initialize Pipeline State and Agents

In [ ]:
from IPython.display import HTML, Markdown, display

from src.agents.multi_genie_orchestrator import MultiGenieOrchestrator
from src.agents.planner_agent import PlannerAgent
from src.agents.report_writer import ReportWriter
from src.agents.synthesizer_agent import SynthesizerAgent
from src.demo import PipelineStage, PipelineState, render_for_notebook

# Initialize fresh pipeline state
state = PipelineState()
state.initialize_spaces([sc.name for sc in space_configs])

# Initialize agents
planner = PlannerAgent(config, space_configs)
orchestrator = MultiGenieOrchestrator(
    space_configs,
    config,
    progress_callback=state.get_progress_callback(),
)
synthesizer = SynthesizerAgent(config)
report_writer = ReportWriter(config)

# Define the question
question = "Generate a Q4 executive report analyzing sales performance, customer trends, and inventory status"

print("Pipeline state and agents initialized.")
print(f"\nQuestion: {question}")
print(f"\nConfigured spaces: {[sc.name for sc in space_configs]}")

### 5.2 Planning Stage

The PlannerAgent decomposes the complex question into domain-specific sub-queries.

In [ ]:
# Execute planning stage
state.start_stage(PipelineStage.PLANNING)

try:
    state.plan = planner.decompose(question)
    state.complete_stage(PipelineStage.PLANNING)

    print("Planning Complete!")
    print("=" * 60)
    print(f"\nOriginal Question: {state.plan.original_question}")
    print(f"\nTarget Spaces: {state.plan.target_spaces}")
    print(f"\nSub-Queries ({len(state.plan.sub_queries)}):")
    for i, sq in enumerate(state.plan.sub_queries, 1):
        print(f"  {i}. [{sq.target_space}] {sq.query}")
    print(f"\nSynthesis Instructions: {state.plan.synthesis_instructions}")

except Exception as e:
    state.fail_stage(PipelineStage.PLANNING, str(e))
    print(f"Planning failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

### 5.3 Querying Stage

The MultiGenieOrchestrator executes parallel queries across all Genie Spaces.

In [ ]:
# Execute querying stage
state.start_stage(PipelineStage.QUERYING)

try:
    state.multi_result = orchestrator.query_all(question)
    state.reconcile_from_result(state.multi_result)
    state.complete_stage(PipelineStage.QUERYING)

    print("Querying Complete!")
    print("=" * 60)

    # Show results summary
    successful = state.multi_result.successful_results()
    failed = state.multi_result.get_failed_spaces()

    print(f"\nSuccessful Queries: {len(successful)}/{len(space_configs)}")
    for name, result in successful.items():
        meta = state.multi_result.metadata.get(name)
        timing = f" ({meta.query_time_seconds:.2f}s)" if meta else ""
        row_count = len(result.data) if result.data else 0
        print(f"  - {name}: {row_count} rows{timing}")

    if failed:
        print(f"\nFailed Queries: {len(failed)}")
        for name in failed:
            result = state.multi_result.results.get(name)
            error = result.error if result else "Unknown error"
            print(f"  - {name}: {error}")

    # Show cache stats if available
    cache_stats = orchestrator.get_cache_stats()
    if cache_stats:
        print(f"\nCache Stats: {cache_stats.get('hits', 0)} hits, {cache_stats.get('misses', 0)} misses")

except Exception as e:
    state.fail_stage(PipelineStage.QUERYING, str(e))
    print(f"Querying failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

In [ ]:
# Display detailed query results
if state.multi_result:
    display(Markdown(state.multi_result.to_combined_markdown(max_rows_per_space=5)))
else:
    print("No query results available. Run the querying stage first.")

### 5.4 Synthesizing Stage

The SynthesizerAgent combines results to generate cross-domain insights.

In [ ]:
# Execute synthesizing stage
state.start_stage(PipelineStage.SYNTHESIZING)

try:
    state.synthesis_result = synthesizer.synthesize(state.multi_result, question)
    state.complete_stage(PipelineStage.SYNTHESIZING)

    print("Synthesis Complete!")
    print("=" * 60)

    result = state.synthesis_result
    print(f"\nDomains Analyzed: {', '.join(result.domains_analyzed)}")

    if result.domains_unavailable:
        print(f"Domains Unavailable: {', '.join(result.domains_unavailable)}")

    print(f"\nKey Insights: {len(result.key_insights)}")
    for i, insight in enumerate(result.key_insights, 1):
        print(f"  {i}. [{insight.importance.upper()}] {insight.insight}")

    print(f"\nCorrelations: {len(result.cross_domain_correlations)}")
    for corr in result.cross_domain_correlations:
        print(f"  - {corr.description}")

    print(f"\nAnomalies: {len(result.anomalies)}")
    for anomaly in result.anomalies:
        print(f"  - [{anomaly.severity.upper()}] {anomaly.description}")

    print(f"\nRecommendations: {len(result.recommendations)}")
    for i, rec in enumerate(result.recommendations, 1):
        print(f"  {i}. {rec}")

except Exception as e:
    state.fail_stage(PipelineStage.SYNTHESIZING, str(e))
    print(f"Synthesis failed: {e}")

# Show progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

### 5.5 Reporting Stage

The ReportWriter generates formatted Markdown and HTML reports.

In [ ]:
# Execute reporting stage
state.start_stage(PipelineStage.REPORTING)

try:
    state.markdown_report = report_writer.generate_markdown(state.synthesis_result, title="Q4 Executive Report")
    state.html_report = report_writer.generate_html(state.synthesis_result, title="Q4 Executive Dashboard")
    state.complete_stage(PipelineStage.REPORTING)

    print("Reporting Complete!")
    print("=" * 60)
    print(f"\nMarkdown report: {len(state.markdown_report)} characters")
    print(f"HTML report: {len(state.html_report)} characters")

except Exception as e:
    state.fail_stage(PipelineStage.REPORTING, str(e))
    print(f"Reporting failed: {e}")

# Show final progress
print("\n" + "=" * 60)
display(Markdown(render_for_notebook(state, include_timing=True)))

## 6. Results Display

View the generated reports.

### HTML Progress Visualization (Phase 2)

Display pipeline progress using HTML progress cards instead of markdown tables.

In [ ]:
# Display HTML progress cards (Phase 2 visualization)
from src.demo import render_progress_html

if state.total_duration_seconds is not None:
    # Get cache stats for display
    cache_stats = orchestrator.get_cache_stats() if "orchestrator" in dir() else None

    # Render HTML progress visualization
    html_progress = render_progress_html(state, cache_stats=cache_stats)
    display(HTML(html_progress))
else:
    print("Pipeline not yet executed. Run End-to-End or Step-Through mode first.")

In [ ]:
# Display the Markdown report
if state.markdown_report:
    display(Markdown(state.markdown_report))
else:
    print("No Markdown report available. Complete the pipeline first.")

In [ ]:
# Display the HTML dashboard (if supported by the notebook environment)
if state.html_report:
    # For Databricks notebooks or Jupyter with HTML support
    display(HTML(state.html_report))
else:
    print("No HTML report available. Complete the pipeline first.")

In [ ]:
# Alternative: Display synthesis result directly using its built-in markdown formatter
if state.synthesis_result:
    display(Markdown(state.synthesis_result.to_markdown()))
else:
    print("No synthesis result available.")

## 7. Cleanup

Reset pipeline state and clear caches for a fresh run.

In [ ]:
# Reset pipeline state
state.reset()
print("Pipeline state reset.")

# Reset orchestrator conversations
orchestrator.reset_all_conversations()
print("Orchestrator conversations reset.")

# Invalidate query cache
invalidated = orchestrator.invalidate_cache()
print(f"Cache invalidated ({invalidated} entries cleared).")

# Clear supervisor history (if used in LangGraph section)
try:
    supervisor.clear_history()
    print("Supervisor history cleared.")
except NameError:
    pass  # supervisor not created in this run

try:
    supervisor_stateful.clear_history()
    print("Stateful supervisor history cleared.")
except NameError:
    pass  # supervisor_stateful not created in this run

# Clear config cache
clear_config_cache()
print("Configuration cache cleared.")

print("\nReady for a fresh run!")

---

## Architecture Summary

This notebook demonstrated two complementary orchestration patterns.

### Pattern 1: Linear Pipeline (Sections 3 & 5)

```
                    User Question
                          |
                          v
                 +----------------+
                 | PlannerAgent   |  Decompose question into
                 | (LLM-powered)  |  domain-specific sub-queries
                 +----------------+
                          |
                          v
          +---------------------------------------+
          |      MultiGenieOrchestrator           |  Parallel execution
          |                                       |  with progress tracking
          +---------------------------------------+
              /           |           \
             v            v            v
        +--------+  +----------+  +----------+
        | Sales  |  | Customers|  | Inventory|
        | Genie  |  | Genie    |  | Genie    |
        +--------+  +----------+  +----------+
              \           |           /
               v          v          v
          +---------------------------------------+
          |        SynthesizerAgent               |  Cross-domain insights
          |        (LLM-powered)                  |  and recommendations
          +---------------------------------------+
                          |
                          v
                 +----------------+
                 | ReportWriter   |  Markdown and HTML
                 | (Jinja2)       |  dashboard generation
                 +----------------+
                          |
                          v
                  Final Reports
```

### Pattern 2: Cyclic Agent Graph (Section 4)

```
          User Message
               |
               v
      +----------------+
  +-->| supervisor     |----> END (respond to user)
  |   | (LLM decides)  |
  |   +----------------+
  |          |
  |          v
  |   +-------------+
  |   | route_next  |  Conditional edge
  |   +-------------+
  |     /         \
  |    v           v
  | +----------+ +---------------+
  | |query_data| |search_documents|
  | | (Genie)  | | (RAG)         |
  | +----------+ +---------------+
  |     \           /
  |      v         v
  |   +-------------+
  +---| after_tools |  Loop back
      +-------------+
```

### When to Use Which

| | Linear Pipeline | Cyclic Agent Graph |
|---|---|---|
| **State** | Stateless (batch) | Stateful (multi-turn) |
| **Flow** | Fixed stage order | Dynamic tool routing |
| **Scope** | All spaces queried every time | Only relevant tools called |
| **Best for** | Scheduled reports, dashboards | Interactive exploration, ad-hoc questions |
| **Complexity** | Simple, predictable | More flexible, harder to debug |

### Key Components

| Component | Purpose | Technology |
|-----------|---------|------------|
| PlannerAgent | Query decomposition | ChatDatabricks LLM |
| MultiGenieOrchestrator | Parallel query execution | ThreadPoolExecutor |
| SupervisorRunner | Cyclic tool-use agent | LangGraph StateGraph |
| SynthesizerAgent | Cross-domain analysis | ChatDatabricks LLM |
| ReportWriter | Report generation | Jinja2 templates |

---

## Next Steps

### Continue Learning

- **Genie SDK Demo**: See the Genie agent in action with the SDK
  - `01_genie_sdk_demo.ipynb`

- **Build Your Own Agent**: Hands-on workshop for building LangGraph agents
  - `03_build_your_agent.ipynb`